# RQ2-v2 Hybrid anchor selector — CPU only

This notebook performs selection and preregistration only. It does not initialize/train a model, download CIFAR-100, or read accuracy, predictions, specialization gaps, or test data. It enumerates all 91 endpoint-locked four-anchor sets, builds the $(R_G,R_R)$ Pareto frontier, applies the frozen normalized-minimax rule, and freezes the seed-3/4/5 protocol.

## Secure repository checkout
Create a Kaggle secret named `github_token`. `KAGGLE_API_TOKEN` is not used. Internet is needed only for cloning the private repository; no accelerator is required.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

## Validate the attached Uniform-100 development artifact

In [ ]:
import importlib
import rq2_anchor_placement
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)

UNIFORM_INPUT_ROOT = Path('/kaggle/input/datasets/dyhngg/100ep-extended')
assert UNIFORM_INPUT_ROOT.exists(), f'Attach dataset: {UNIFORM_INPUT_ROOT}'
UNIFORM_ROOT = rq2_anchor_placement.find_uniform_100_root(
    UNIFORM_INPUT_ROOT, '/kaggle/working/materialized-uniform-100-hybrid'
)
print('Validated Uniform-100 root:', UNIFORM_ROOT)

## Enumerate, select and freeze
Geometry is the edge-wise median of Uniform-100 learned-projection validation geometry over development seeds 0,1,2. Resource position is min-max normalized log FLOPs.

In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display

OUTPUT_DIR = Path('/kaggle/working/hybrid-v2-selection')
started = time.perf_counter()
selection = rq2_anchor_placement.select_hybrid_anchors(UNIFORM_ROOT, OUTPUT_DIR)
print(f'Selection completed in {time.perf_counter() - started:.2f} seconds')
print(json.dumps(selection, indent=2))

## Inspect selection, Pareto frontier and frozen gates

In [ ]:
candidates = pd.read_csv(OUTPUT_DIR / 'hybrid_anchor_candidates.csv')
frontier = pd.read_csv(OUTPUT_DIR / 'hybrid_pareto_frontier.csv')
compute = pd.read_csv(OUTPUT_DIR / 'hybrid_training_compute.csv')
display(Markdown((OUTPUT_DIR / 'hybrid_v2_selection_report.md').read_text()))
display(Markdown('### Selected Hybrid-v2 candidate'))
display(candidates[candidates.selected])
display(Markdown('### Pareto frontier'))
display(frontier)
display(Markdown('### State-count and FLOPs diagnostic'))
display(compute)
display(Markdown('### Frozen confirmatory gates'))
display(json.loads((OUTPUT_DIR / 'hybrid_v2_preregistered_gates.json').read_text()))
display(Image(filename=str(OUTPUT_DIR / 'hybrid_anchor_pareto.png')))

## Validate and export the CPU-selection bundle

In [ ]:
REQUIRED = [
    'hybrid_anchor_candidates.csv',
    'hybrid_pareto_frontier.csv',
    'hybrid_coordinates.csv',
    'hybrid_training_compute.csv',
    'hybrid_selected_anchors.json',
    'hybrid_v2_preregistered_gates.json',
    'hybrid_anchor_pareto.png',
    'hybrid_v2_selection_report.md',
]
for filename in REQUIRED:
    path = OUTPUT_DIR / filename
    assert path.is_file() and path.stat().st_size > 0, f'Missing/empty output: {path}'
assert len(pd.read_csv(OUTPUT_DIR / 'hybrid_anchor_candidates.csv')) == 91
assert int(pd.read_csv(OUTPUT_DIR / 'hybrid_anchor_candidates.csv')['selected'].sum()) == 1
bundle_path = Path('/kaggle/working/rq2-hybrid-v2-cpu-selection.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
print('Download:', bundle_path)
bundle_path